In [ ]:
"""
=============================================================================
[Advanced Code Architecture: Transformer & Beam Search Optimization]

1. Architecture Upgrade (Transformer):
   - Replaced Bi-GRU with Seq2Seq Transformer (Encoder-Decoder).
   - Core: Multi-Head Self-Attention + Positional Encoding.
   - Optimization: Pre-Norm (norm_first=True) for better gradient flow.

2. Semantic Enhancement:
   - Tokenization: Retained Char-level (CN) / Space-split (EN) to remove external dependencies.
   - Pretrained Embeddings: Interface added via `EmbeddingsLoader`.

3. Decoding Strategy (Beam Search):
   - Algorithm: Top-K Beam Search with Length Normalization.
   - Efficiency: Caches Encoder outputs to avoid redundant computation.

4. OOP Refactoring:
   - Strict typing and modular design.

5. Hyperparameter Optimization (Optuna):
   - Search Space: d_model, nhead, num_layers, dim_feedforward, dropout.
   - Constraint Handling: Ensures d_model is divisible by nhead.

6. Strict Experimental Setup:
   - Restored full pipeline: Data -> Optuna -> Train -> Plot -> Validation -> Test File.
=============================================================================
"""

import unicodedata
import string
import re
import random
import time
import math
import os
import sys
import warnings
import numpy as np
import matplotlib.pyplot as plt
from typing import Tuple, List, Optional, Dict, Any

import torch
import torch.nn as nn
from torch import optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

warnings.filterwarnings(
    "ignore", category=UserWarning, module="torch.nn.modules.transformer"
)

import optuna
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction, corpus_bleu


# =============================================================================
# 0. Hardware & System Setup
# =============================================================================
def check_gpu() -> None:
    if not torch.cuda.is_available():
        print("\n" + "=" * 60)
        print("CRITICAL ERROR: GPU NOT DETECTED!")
        sys.exit(1)

    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print(f"\n[Hardware] GPU: {torch.cuda.get_device_name(0)}")
    print("[Hardware] TF32 & AMP Enabled.\n")


check_gpu()


# =============================================================================
# 1. Configuration
# =============================================================================
class Config:
    SEED = 2025
    DEVICE = torch.device("cuda")

    # Data Settings
    LANG_SRC = "cn"
    LANG_TGT = "eng"
    MAX_LENGTH = 100
    SOS_TOKEN = 0
    EOS_TOKEN = 1
    PAD_TOKEN = 2

    BATCH_SIZE = 128
    NUM_WORKERS = 4
    PIN_MEMORY = True

    # Training Settings
    N_EPOCHS_OPT = 20
    N_EPOCHS_FULL = 200
    CLIP = 1.0
    PRINT_EVERY = 5
    PATIENCE = 15
    ACCUMULATE_GRAD_STEPS = 1

    # Optimizer Settings
    LEARNING_RATE = 1e-4
    WEIGHT_DECAY = 1e-4
    LABEL_SMOOTHING = 0.1

    # Optuna Settings
    N_TRIALS = 200


class Utils:
    @staticmethod
    def seed_everything(seed: int) -> None:
        random.seed(seed)
        os.environ["PYTHONHASHSEED"] = str(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        print(f"[System] Seed: {seed} | Device: {Config.DEVICE}")

    @staticmethod
    def as_minutes(s: float) -> str:
        m = math.floor(s / 60)
        s -= m * 60
        return "%dm %ds" % (m, s)

    @staticmethod
    def time_since(start: float, percent: float) -> str:
        now = time.time()
        s = now - start
        if percent == 0:
            return "0m 0s"
        es = s / percent
        rs = es - s
        return "%s (- %s)" % (Utils.as_minutes(s), Utils.as_minutes(rs))


Utils.seed_everything(Config.SEED)


# =============================================================================
# 2. Data Processing (No Jieba, Pure Python)
# =============================================================================
class Vocab:
    def __init__(self, name: str):
        self.name = name
        self.word2index = {}
        self.word2count = {}
        self.index2word = {0: "SOS", 1: "EOS", 2: "PAD"}
        self.n_words = 3

    def add_sentence(self, sentence: str):
        iterator = sentence if self.name == "cn" else sentence.split(" ")
        for word in iterator:
            self.add_word(word)

    def add_word(self, word: str):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1


class DataEngine:
    @staticmethod
    def unicode_to_ascii(s: str) -> str:
        return "".join(
            c
            for c in unicodedata.normalize("NFD", s)
            if unicodedata.category(c) != "Mn"
        )

    @staticmethod
    def normalize_string(s: str) -> str:
        s = DataEngine.unicode_to_ascii(s.lower().strip())
        s = re.sub(r"([.!?])", r" \1", s)
        s = re.sub(r"[^a-zA-Z\u4e00-\u9fa5.!?，。？]+", r" ", s)
        return s.strip()

    @staticmethod
    def prepare_data(lang1: str, lang2: str) -> Tuple[Vocab, Vocab, List[List[str]]]:
        filename = f"{lang1}-{lang2}.txt"
        if not os.path.exists(filename):
            print(f"[Warning] {filename} not found. Creating dummy data.")
            lines = [
                "你好\thello",
                "谢谢\tthank you",
                "再见\tgoodbye",
                "我爱你\ti love you",
            ] * 200
        else:
            lines = open(filename, encoding="utf-8").read().strip().split("\n")

        pairs = [
            [DataEngine.normalize_string(s) for s in line.split("\t")] for line in lines
        ]
        pairs = [
            p
            for p in pairs
            if len(p[0]) < Config.MAX_LENGTH
            and len(p[1].split(" ")) < Config.MAX_LENGTH
        ]

        input_lang = Vocab(lang1)
        output_lang = Vocab(lang2)

        for pair in pairs:
            input_lang.add_sentence(pair[0])
            output_lang.add_sentence(pair[1])

        print(
            f"[Data] Loaded {len(pairs)} pairs. Vocab: {input_lang.n_words}/{output_lang.n_words}"
        )
        return input_lang, output_lang, pairs

    @staticmethod
    def collate_fn(batch):
        # Output: (Seq_Len, Batch_Size) for Transformer compatibility (S, N)
        input_batch, target_batch = zip(*batch)
        input_pad = pad_sequence(
            input_batch, padding_value=Config.PAD_TOKEN, batch_first=False
        )
        target_pad = pad_sequence(
            target_batch, padding_value=Config.PAD_TOKEN, batch_first=False
        )
        return input_pad, target_pad


class TranslationDataset(Dataset):
    def __init__(self, pairs, input_lang, output_lang, augment=False):
        self.pairs = pairs
        self.input_lang = input_lang
        self.output_lang = output_lang
        self.augment = augment

    def __len__(self):
        return len(self.pairs)

    def augment_sequence(self, indices: List[int]) -> List[int]:
        if len(indices) <= 3:
            return indices
        new_indices = indices.copy()
        # Random Swap
        if random.random() < 0.15:
            i1, i2 = random.sample(range(len(new_indices)), 2)
            new_indices[i1], new_indices[i2] = new_indices[i2], new_indices[i1]
        # Random Drop
        if random.random() < 0.1:
            del new_indices[random.randint(0, len(new_indices) - 1)]
        return new_indices

    def __getitem__(self, idx):
        pair = self.pairs[idx]
        if self.input_lang.name == "cn":
            input_indices = [self.input_lang.word2index[w] for w in pair[0]]
        else:
            input_indices = [self.input_lang.word2index[w] for w in pair[0].split(" ")]

        if self.augment:
            input_indices = self.augment_sequence(input_indices)

        input_indices.append(Config.EOS_TOKEN)

        if self.output_lang.name == "cn":
            target_indices = [self.output_lang.word2index[w] for w in pair[1]]
        else:
            target_indices = [
                self.output_lang.word2index[w] for w in pair[1].split(" ")
            ]
        target_indices = (
            [Config.SOS_TOKEN] + target_indices + [Config.EOS_TOKEN]
        )  # Prepend SOS for decoder

        return torch.tensor(input_indices, dtype=torch.long), torch.tensor(
            target_indices, dtype=torch.long
        )


# =============================================================================
# 3. Transformer Model
# =============================================================================


class EmbeddingsLoader:
    """Interface for loading pre-trained embeddings (Word2Vec/GloVe)."""

    @staticmethod
    def load_pretrained(vocab: Vocab, emb_dim: int, path: str = None) -> torch.Tensor:
        embedding_matrix = torch.zeros((vocab.n_words, emb_dim))
        nn.init.normal_(embedding_matrix, mean=0, std=emb_dim**-0.5)

        if path and os.path.exists(path):
            print(f"[Embeddings] Loading external vectors from {path}...")
            # Logic to parse file and match words in vocab would go here
            pass

        return embedding_matrix


class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )
        pe = torch.zeros(max_len, 1, d_model)
        pe[:, 0, 0::2] = torch.sin(position * div_term)
        pe[:, 0, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [SeqLen, Batch, Dim]
        x = x + self.pe[: x.size(0)]
        return self.dropout(x)


class Seq2SeqTransformer(nn.Module):
    def __init__(
        self,
        src_vocab_size: int,
        tgt_vocab_size: int,
        d_model: int,
        nhead: int,
        num_encoder_layers: int,
        num_decoder_layers: int,
        dim_feedforward: int,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.d_model = d_model

        self.src_embedding = nn.Embedding(src_vocab_size, d_model)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, dropout=dropout)

        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=False,
            norm_first=True,
        )

        self.generator = nn.Linear(d_model, tgt_vocab_size)
        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def load_embeddings(self, src_emb: torch.Tensor, tgt_emb: torch.Tensor):
        self.src_embedding.weight.data.copy_(src_emb)
        self.tgt_embedding.weight.data.copy_(tgt_emb)

    def generate_square_subsequent_mask(self, sz: int) -> torch.Tensor:
        mask = torch.triu(torch.ones((sz, sz), device=Config.DEVICE), diagonal=1).bool()
        return mask

    def create_mask(self, src: torch.Tensor, tgt: torch.Tensor):
        # src: [Seq, Batch], tgt: [Seq, Batch]
        src_seq_len = src.shape[0]
        tgt_seq_len = tgt.shape[0]

        tgt_mask = self.generate_square_subsequent_mask(tgt_seq_len)

        src_mask = torch.zeros((src_seq_len, src_seq_len), device=Config.DEVICE).bool()

        src_padding_mask = (src == Config.PAD_TOKEN).transpose(0, 1)  # [Batch, Seq]
        tgt_padding_mask = (tgt == Config.PAD_TOKEN).transpose(0, 1)  # [Batch, Seq]

        return src_mask, tgt_mask, src_padding_mask, tgt_padding_mask

    def forward(self, src: torch.Tensor, tgt: torch.Tensor):
        # src: [S, N], tgt: [T, N]
        src_mask, tgt_mask, src_padding_mask, tgt_padding_mask = self.create_mask(
            src, tgt
        )

        src_emb = self.positional_encoding(
            self.src_embedding(src) * math.sqrt(self.d_model)
        )
        tgt_emb = self.positional_encoding(
            self.tgt_embedding(tgt) * math.sqrt(self.d_model)
        )

        outs = self.transformer(
            src_emb,
            tgt_emb,
            src_mask=src_mask,
            tgt_mask=tgt_mask,
            memory_mask=None,
            src_key_padding_mask=src_padding_mask,
            tgt_key_padding_mask=tgt_padding_mask,
            memory_key_padding_mask=src_padding_mask,
        )

        return self.generator(outs)

    def encode(self, src: torch.Tensor, src_mask: torch.Tensor):
        src_padding_mask = (src == Config.PAD_TOKEN).transpose(0, 1)
        src_emb = self.positional_encoding(
            self.src_embedding(src) * math.sqrt(self.d_model)
        )
        return self.transformer.encoder(src_emb, src_key_padding_mask=src_padding_mask)

    def decode(self, tgt: torch.Tensor, memory: torch.Tensor, src_mask: torch.Tensor):
        tgt_seq_len = tgt.shape[0]
        tgt_mask = self.generate_square_subsequent_mask(tgt_seq_len)
        tgt_emb = self.positional_encoding(
            self.tgt_embedding(tgt) * math.sqrt(self.d_model)
        )

        return self.transformer.decoder(tgt_emb, memory, tgt_mask=tgt_mask)


# =============================================================================
# 4. Evaluator (Beam Search)
# =============================================================================
class BeamNode:
    def __init__(self, sequence: List[int], log_prob: float, length: int):
        self.sequence = sequence
        self.log_prob = log_prob
        self.length = length

    def score(self, alpha: float = 0.7):
        return self.log_prob / (self.length**alpha)


class Evaluator:
    @staticmethod
    def beam_search(
        model: Seq2SeqTransformer,
        src_tensor: torch.Tensor,
        output_lang: Vocab,
        beam_width: int = 5,
        max_len: int = 100,
    ) -> List[str]:
        """
        Implements Beam Search for Transformer.
        """
        model.eval()
        with torch.no_grad():
            memory = model.encode(src_tensor, None)  # [S, 1, E]

            # Start with SOS
            start_node = BeamNode(sequence=[Config.SOS_TOKEN], log_prob=0.0, length=1)
            beam = [start_node]

            finished_beams = []

            for _ in range(max_len):
                candidates = []

                # Expand valid beams
                for node in beam:
                    if node.sequence[-1] == Config.EOS_TOKEN:
                        finished_beams.append(node)
                        continue

                    tgt_inp = torch.tensor(
                        node.sequence, dtype=torch.long, device=Config.DEVICE
                    ).unsqueeze(1)  # [Seq, 1]

                    # Decoder Step
                    out = model.decode(tgt_inp, memory, None)  # [Seq, 1, E]
                    out = model.generator(out)  # [Seq, 1, Vocab]

                    # Get Log Probs for the last token
                    log_probs = F.log_softmax(out[-1, 0, :], dim=0)

                    # Get top K
                    topk_probs, topk_ids = torch.topk(log_probs, beam_width)

                    for k in range(beam_width):
                        idx = topk_ids[k].item()
                        prob = topk_probs[k].item()

                        new_seq = node.sequence + [idx]
                        new_node = BeamNode(
                            new_seq, node.log_prob + prob, node.length + 1
                        )
                        candidates.append(new_node)

                # If all beams finished
                if not candidates:
                    break

                # Sort candidates by normalized score
                candidates.sort(key=lambda x: x.score(), reverse=True)
                beam = candidates[:beam_width]

                # Early stop if we have enough finished beams better than current best candidate
                if len(finished_beams) >= beam_width:
                    # simplistic check
                    break

            # Combine unfinished and finished
            all_nodes = finished_beams + beam
            all_nodes.sort(key=lambda x: x.score(), reverse=True)
            best_seq = all_nodes[0].sequence

            decoded_words = []
            for idx in best_seq:
                if idx == Config.SOS_TOKEN:
                    continue
                if idx == Config.EOS_TOKEN:
                    break
                decoded_words.append(output_lang.index2word[idx])

            return decoded_words

    @staticmethod
    def calculate_bleu(model, pairs, input_lang, output_lang, sample_size=300):
        model.eval()
        check_pairs = random.sample(pairs, min(len(pairs), sample_size))
        refs, cands = [], []

        for pair in check_pairs:
            src_text = pair[0]
            if input_lang.name == "cn":
                src_idxs = [input_lang.word2index[w] for w in src_text]
            else:
                src_idxs = [input_lang.word2index[w] for w in src_text.split(" ")]

            src_tensor = torch.tensor(
                src_idxs, dtype=torch.long, device=Config.DEVICE
            ).unsqueeze(1)

            # Use Greedy search for speed during Validation
            pred_tokens = Evaluator.beam_search(
                model, src_tensor, output_lang, beam_width=1
            )

            refs.append([pair[1].split(" ")])  # Reference needs to be list of lists
            cands.append(pred_tokens)

        return corpus_bleu(refs, cands, smoothing_function=SmoothingFunction().method1)

    @staticmethod
    def predict_test_file(filename, model, input_lang, output_lang):
        print("\n" + "=" * 60)
        print(f"[Test] Processing Test File: {filename}")

        if not os.path.exists(filename):
            print(f"[Error] {filename} not found.")
            return

        with open(filename, "r", encoding="utf-8") as f:
            lines = f.read().strip().split("\n")

        lines = [line for line in lines if line.strip()]
        total_lines = len(lines)

        num_samples = 5
        sample_indices = set(
            random.sample(range(total_lines), min(total_lines, num_samples))
        )

        output_filename = "test_results.txt"
        print(f"[Test] Total lines: {total_lines}")
        print(f"[Test] Saving results to: {output_filename}")

        model.eval()

        with open(output_filename, "w", encoding="utf-8") as out_f:
            for i, line in enumerate(lines):
                src_sentence = DataEngine.normalize_string(line)

                if input_lang.name == "cn":
                    src_idxs = [
                        input_lang.word2index.get(w, Config.PAD_TOKEN)
                        for w in src_sentence
                    ]
                else:
                    src_idxs = [
                        input_lang.word2index.get(w, Config.PAD_TOKEN)
                        for w in src_sentence.split(" ")
                    ]

                src_tensor = torch.tensor(
                    src_idxs, dtype=torch.long, device=Config.DEVICE
                ).unsqueeze(1)

                try:
                    output_words = Evaluator.beam_search(
                        model, src_tensor, output_lang, beam_width=5
                    )

                    output_sentence = " ".join(output_words)

                    out_f.write(f"{line.strip()}\t{output_sentence}\n")

                    if i in sample_indices:
                        print(f"[{i}] Src: {line.strip()}")
                        print(f"    Out: {output_sentence}")
                        print("-" * 30)

                except Exception as e:
                    err_msg = f"Error: {str(e)}"
                    out_f.write(f"{line.strip()}\t{err_msg}\n")

        print("[Test] Translation finished. Results saved.")
        print("=" * 60 + "\n")


# =============================================================================
# 5. Trainer
# =============================================================================
class Trainer:
    def __init__(self, model, lr, weight_decay):
        self.model = model
        self.optimizer = optim.AdamW(
            model.parameters(), lr=lr, weight_decay=weight_decay
        )
        self.criterion = nn.CrossEntropyLoss(
            ignore_index=Config.PAD_TOKEN, label_smoothing=Config.LABEL_SMOOTHING
        )
        self.scaler = torch.amp.GradScaler("cuda")
        self.scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer, mode="max", factor=0.5, patience=5
        )

    def train_epoch(self, loader):
        self.model.train()
        total_loss = 0

        for i, (src, tgt) in enumerate(loader):
            src, tgt = src.to(Config.DEVICE), tgt.to(Config.DEVICE)

            tgt_input = tgt[:-1, :]
            tgt_output = tgt[1:, :]

            self.optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast("cuda"):
                logits = self.model(src, tgt_input)  # [T-1, N, Vocab]

                # Flatten for loss
                output_dim = logits.shape[-1]
                loss = self.criterion(
                    logits.reshape(-1, output_dim), tgt_output.reshape(-1)
                )

            self.scaler.scale(loss).backward()

            if (i + 1) % Config.ACCUMULATE_GRAD_STEPS == 0:
                self.scaler.unscale_(self.optimizer)
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), Config.CLIP)
                self.scaler.step(self.optimizer)
                self.scaler.update()

            total_loss += loss.item()

        return total_loss / len(loader)

    def fit(
        self,
        train_pairs,
        val_pairs,
        input_lang,
        output_lang,
        epochs,
        trial=None,
        verbose=True,
        save_path=None,
    ):
        ds = TranslationDataset(train_pairs, input_lang, output_lang, augment=True)
        loader = DataLoader(
            ds,
            batch_size=Config.BATCH_SIZE,
            shuffle=True,
            num_workers=Config.NUM_WORKERS,
            pin_memory=Config.PIN_MEMORY,
            collate_fn=DataEngine.collate_fn,
        )

        best_bleu = -1.0
        patience_counter = 0
        history = {"loss": [], "bleu": []}
        start = time.time()

        for epoch in range(1, epochs + 1):
            loss = self.train_epoch(loader)
            bleu = Evaluator.calculate_bleu(
                self.model, val_pairs, input_lang, output_lang
            )

            self.scheduler.step(bleu)

            history["loss"].append(loss)
            history["bleu"].append(bleu)

            if verbose and (epoch % Config.PRINT_EVERY == 0 or epoch == epochs):
                print(
                    f"{Utils.time_since(start, epoch / epochs)} (Ep {epoch}) "
                    f"Loss: {loss:.4f} | BLEU: {bleu:.4f} | LR: {self.optimizer.param_groups[0]['lr']:.6f}"
                )

            # Optuna Pruning
            if trial:
                trial.report(bleu, epoch)
                if trial.should_prune():
                    raise optuna.exceptions.TrialPruned()

            if bleu > best_bleu:
                best_bleu = bleu
                patience_counter = 0
                if save_path:
                    torch.save(self.model.state_dict(), save_path)
                    if verbose:
                        print(f"  >>> [Model Saved] New Best BLEU: {bleu:.4f}")
                elif verbose:
                    torch.save(self.model.state_dict(), "best_transformer.pth")
            else:
                patience_counter += 1

            if Config.PATIENCE and patience_counter >= Config.PATIENCE:
                if verbose:
                    print(
                        f"\n[Early Stopping] Triggered! No improvement for {Config.PATIENCE} epochs."
                    )
                break

        return history


# =============================================================================
# 6. Main & Optuna Optimization
# =============================================================================
def objective(trial, input_lang, output_lang, train_pairs, val_pairs):
    nhead = trial.suggest_categorical("nhead", [4, 8])

    base_dim = trial.suggest_int("base_dim", 16, 128, step=16)
    d_model = base_dim * nhead

    num_layers = trial.suggest_int("num_layers", 1, 8)
    dim_feedforward = trial.suggest_int("dim_feedforward", 128, 2048, step=128)
    dropout = trial.suggest_float("dropout", 0.1, 0.8)
    lr = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)

    model = Seq2SeqTransformer(
        src_vocab_size=input_lang.n_words,
        tgt_vocab_size=output_lang.n_words,
        d_model=d_model,
        nhead=nhead,
        num_encoder_layers=num_layers,
        num_decoder_layers=num_layers,
        dim_feedforward=dim_feedforward,
        dropout=dropout,
    ).to(Config.DEVICE)

    trainer = Trainer(model, lr=lr, weight_decay=weight_decay)

    history = trainer.fit(
        train_pairs,
        val_pairs,
        input_lang,
        output_lang,
        epochs=Config.N_EPOCHS_OPT,
        trial=trial,
        verbose=False,
    )
    return max(history["bleu"]) if history["bleu"] else 0


def main():
    # 1. Prepare Data
    input_lang, output_lang, pairs = DataEngine.prepare_data(
        Config.LANG_SRC, Config.LANG_TGT
    )

    random.shuffle(pairs)
    split_at = int(len(pairs) * 0.8)
    train_pairs, val_pairs = pairs[:split_at], pairs[split_at:]

    # 2. Optuna Search
    print(f"\n[Optuna] Starting Massive Search (Trials: {Config.N_TRIALS})...")
    study = optuna.create_study(direction="maximize")
    study.optimize(
        lambda t: objective(t, input_lang, output_lang, train_pairs, val_pairs),
        n_trials=Config.N_TRIALS,
    )

    print(f"[Optuna] Best Params: {study.best_params}")

    best = study.best_params
    # Reconstruct derived params for Transformer
    d_model = best["base_dim"] * best["nhead"]

    # 3. Final Training with Best Params
    final_model = Seq2SeqTransformer(
        input_lang.n_words,
        output_lang.n_words,
        d_model=d_model,
        nhead=best["nhead"],
        num_encoder_layers=best["num_layers"],
        num_decoder_layers=best["num_layers"],
        dim_feedforward=best["dim_feedforward"],
        dropout=best["dropout"],
    ).to(Config.DEVICE)

    trainer = Trainer(final_model, lr=best["lr"], weight_decay=best["weight_decay"])

    print(
        f"\n[Main] Final Training (Epochs: {Config.N_EPOCHS_FULL}) with Early Stopping..."
    )

    # Train with Early Stopping and Model Saving enabled
    hist = trainer.fit(
        train_pairs,
        val_pairs,
        epochs=Config.N_EPOCHS_FULL,
        input_lang=input_lang,
        output_lang=output_lang,
        save_path="best_model.pth",
    )

    # 4. Save Training Plot
    plt.figure(figsize=(10, 4), dpi=300)
    plt.subplot(1, 2, 1)
    plt.plot(hist["loss"], label="Loss")
    plt.legend()
    plt.subplot(1, 2, 2)
    plt.plot(hist["bleu"], label="BLEU", c="orange")
    plt.legend()
    plt.savefig("result_final.png")
    print("[Main] Results saved.")

    # 5. Validation Samples
    print("\n[Validation] Check Samples (using Best Saved Model):")
    final_model.load_state_dict(torch.load("best_model.pth"))

    for _ in range(3):
        p = random.choice(val_pairs)
        try:
            src_text = p[0]
            if input_lang.name == "cn":
                src_idxs = [input_lang.word2index[w] for w in src_text]
            else:
                src_idxs = [input_lang.word2index[w] for w in src_text.split(" ")]

            src_tensor = torch.tensor(
                src_idxs, dtype=torch.long, device=Config.DEVICE
            ).unsqueeze(1)

            res = Evaluator.beam_search(
                final_model, src_tensor, output_lang, beam_width=5
            )
            print(f"SRC: {p[0]}\nTGT: {p[1]}\nOUT: {' '.join(res)}\n---")
        except KeyError:
            pass

    # 6. Final Test on 'test.txt'
    Evaluator.predict_test_file("test.txt", final_model, input_lang, output_lang)


if __name__ == "__main__":
    main()